In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/cod-reproduction')
ROOT.mkdir(parents=True, exist_ok=True)
PAYLOAD = '...'
bundle = base64.b64decode(PAYLOAD)
assert hashlib.sha256(bundle).hexdigest() == 'e75d84ae7599f18ed4599f6872bf10f8d70b6efa0cc7121a94e2cd7026980f6e'
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    archive.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML==6.0.3', 'transformers==4.57.1', 'accelerate==1.10.1'], check=True)
import torch
print('Python', sys.version.split()[0], 'torch', torch.__version__)
print('GPUs', [(torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1)) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), 'Enable GPU T4 x2 and Internet before running'
from prepare_resources import verify_local_resources
verify_local_resources()


In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover(str(ROOT), pattern='test_experiment.py')
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), 'Focused validation failed'


In [ ]:
from experiment import run_experiment
pilot_summary = run_experiment(ROOT / 'config.json', 'pilot', ROOT / 'runs/pilot')
print(json.dumps(pilot_summary, ensure_ascii=False, indent=2))


In [ ]:
full_summary = run_experiment(ROOT / 'config.json', 'full', ROOT / 'runs/full')
print(json.dumps(full_summary, ensure_ascii=False, indent=2))


In [ ]:
with (ROOT / 'runs/requirements-freeze.txt').open('w', encoding='utf-8') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=handle, check=True)
with zipfile.ZipFile('/kaggle/working/cod-results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in (ROOT / 'runs').rglob('*'):
        if path.is_file():
            archive.write(path, path.relative_to(ROOT))
print('Saved /kaggle/working/cod-results.zip')
